In [5]:
from functools import partial

import astroplan as ap
import astropy.units as u
import numpy as np
import pandas as pd

from astropaul.database import database_connection
import astropaul.targetlistcreator as tlc
import astropaul.lbt as lbt
import astropaul.html as html
import astropaul.phase as ph
import astropaul.priority as pr

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
session = tlc.ObservingSession(ap.Observer.at_site("LBT"))
session.add_day_range("2026-07-01", "2026-12-31")
proposal_cycle = "2026B"

pepsi_args = {
    "fiber": "300",
    "cd_blue": 3,
    "cd_red": 6,
    "snr": 100,
    "binocular": True,
    "priority": "(see grid)",
}

name = f"LBT Proposal {proposal_cycle}"
with database_connection() as conn:
    creator = tlc.TargetListCreator(name=name, connection=conn)
    creator.steps = [
        tlc.add_targets,
        partial(tlc.filter_targets, criteria=lambda df: (df["Target Type"].isin(["QuadEB", "SextEB"]))),
        tlc.ancillary_data_from_tess,
        partial(tlc.add_database_table, table_name="Ephemerides"),
        partial(tlc.add_database_table, table_name="PEPSI Observations"),
        partial(tlc.filter_targets, criteria=lambda df: (df["Num PEPSI Observations"] > 0)),
        partial(tlc.add_observability, observing_session=session, time_resolution=2 * u.hour),
        partial(tlc.filter_targets, criteria=lambda df: (df["Observable Any Night"])),
        partial(tlc.filter_targets, inverse=True, criteria=lambda df: df["Teff"].isna()),
        partial(lbt.add_pepsi_params, **pepsi_args),
        partial(tlc.filter_targets, criteria=lambda df: df["PEPSI exp_time"] < 600),
    ]
    tl = creator.calculate(verbose=False)

tl.target_list["PEPSI exp_time"] += 120 # add 2 minutes overhead to each target

print(tl.summarize())
print(f'{np.sum(tl.target_list["PEPSI exp_time"])/60:.1f} minutes')
# tl.target_list

/Users/paul/Library/CloudStorage/Dropbox/Astro/astropaul/astropaul/targetlistcreator/observability.py:95: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  answer.target_list = answer.target_list.assign(**new_cols)
/Users/paul/Library/CloudStorage/Dropbox/Astro/astropaul/astropaul/targetlistcreator/observability.py:95: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  answer.target_list = answer.target_list.assign(**new_cols)
/Users/paul/Library/CloudStorage/Dropbox/Astro/astropaul/astropaul/targetlistcreator/observability.py:95: Perfor

Name: LBT Proposal 2026B
Criteria:
  All targets loaded (732 targets remain)
  lambda df: (df["Target Type"].isin(["QuadEB", "SextEB"])) (322 targets remain)
  lambda df: (df["Num PEPSI Observations"] > 0) (31 targets remain)
  Observability calculated at LBT in 2.0 h intervals from 2026-07-01 to 2026-12-31
    AltitudeConstraint: {'min': np.float64(30.0), 'max': np.float64(80.0), 'boolean_constraint': True}
  lambda df: (df["Observable Any Night"]) (31 targets remain)
  Inverse of: lambda df: df["Teff"].isna() (24 targets remain)
  lambda df: df["PEPSI exp_time"] < 600 (22 targets remain)
  
22 targets:
    22 QuadEB
Column Count (primary, secondary):
    Target: (3, 4)
    TESS Data: (4, 0)
    Count: (2, 0)
    Observable: (4, 552)
    PEPSI : (3, 6)
Associated tables:
     733 rows, 126 columns: TESS
     894 rows,   7 columns: Ephemerides
     289 rows,  14 columns: PEPSI Observations
     184 rows,   2 columns: Lunar Phases

Total PEPSI exposure time: 111.5 minutes
111.5 minutes


In [7]:
tl.target_list

,Target Name,RA,Dec,RA HMS,Dec DMS,Target Type,Target Source,Vmag,Teff,PMra,...,Observable Hours Min,PEPSI fiber,PEPSI cd_blue,PEPSI cd_blue_num_exp,PEPSI cd_red,PEPSI cd_red_num_exp,PEPSI snr,PEPSI exp_time,PEPSI priority,PEPSI notes
225,TIC 123098844,279.572833,44.698600,18:38:17.48,+44:41:54.96,QuadEB,Kostov 2022 doi.org/10.3847/1538-4365/ac5458,11.136,6761.00,3.482500,...,2.0,300,3,1,6,1,100,309,(see grid),
240,TIC 204698586,185.551833,-24.224844,12:22:12.44,-24:13:29.44,QuadEB,Kostov 2022 doi.org/10.3847/1538-4365/ac5458,11.506,6392.00,1.157790,...,0.0,300,3,1,6,1,100,393,(see grid),
250,TIC 25818450,352.743458,53.069150,23:30:58.43,+53:04:08.94,QuadEB,Kostov 2022 doi.org/10.3847/1538-4365/ac5458,11.782,7172.00,3.573600,...,4.0,300,3,1,6,1,100,461,(see grid),
251,TIC 260056937,63.922208,47.422197,04:15:41.33,+47:25:19.91,QuadEB,Kostov 2022 doi.org/10.3847/1538-4365/ac5458,10.302,8030.00,1.294310,...,0.0,300,3,1,6,1,100,201,(see grid),
260,TIC 278352276,307.503625,48.607056,20:30:00.87,+48:36:25.4,QuadEB,Kostov 2022 doi.org/10.3847/1538-4365/ac5458,10.387,7156.00,1.853870,...,4.0,300,3,1,6,1,100,214,(see grid),
261,TIC 283940788,8.851417,62.901594,00:35:24.34,+62:54:05.74,QuadEB,Kostov 2022 doi.org/10.3847/1538-4365/ac5458,11.774,7384.00,-2.726640,...,4.0,300,3,1,6,1,100,448,(see grid),
265,TIC 286470992,45.330708,60.572294,03:01:19.37,+60:34:20.26,QuadEB,Kostov 2022 doi.org/10.3847/1538-4365/ac5458,10.325,8693.00,0.149592,...,2.0,300,3,1,6,1,100,201,(see grid),
269,TIC 307119043,14.827542,51.221642,00:59:18.61,+51:13:17.91,QuadEB,Kostov 2022 doi.org/10.3847/1538-4365/ac5458,9.940,7815.00,7.560190,...,4.0,300,3,1,6,1,100,181,(see grid),
274,TIC 317863971,110.567500,3.031925,07:22:16.2,+03:01:54.93,QuadEB,Kostov 2022 doi.org/10.3847/1538-4365/ac5458,10.306,8506.00,-1.844620,...,0.0,300,3,1,6,1,100,201,(see grid),
277,TIC 322727163,309.716625,50.466819,20:38:51.99,+50:28:00.55,QuadEB,Kostov 2022 doi.org/10.3847/1538-4365/ac5458,10.997,7876.56,2.800000,...,4.0,300,3,1,6,1,100,274,(see grid),


In [8]:
# make a CSV file for use in the PIT
column_map = [
    ("Name", "Target Name"), 
    ("RAJ2000", "RA"),
    ("DecJ2000", "Dec"),
    ("V", "Vmag"),
    ("pepsi_exposure_time", "PEPSI exp_time"),
]

proposal_table = pd.DataFrame()
for csv_name, tl_name in column_map:
    proposal_table[csv_name] = tl.target_list[tl_name]
proposal_table.to_csv(f"LBT {proposal_cycle} Proposal Targets.csv", index=False)

In [9]:
foo = pd.read_csv(f"LBT {proposal_cycle} Proposal Targets.csv")
foo["hours"] = foo["pepsi_exposure_time"] / 3600
np.min(foo["pepsi_exposure_time"]) / 3600
foo.sort_values("Name")

,Name,RAJ2000,DecJ2000,V,pepsi_exposure_time,hours
0,TIC 123098844,279.572833,44.698600,11.136,309,0.085833
1,TIC 204698586,185.551833,-24.224844,11.506,393,0.109167
2,TIC 25818450,352.743458,53.069150,11.782,461,0.128056
3,TIC 260056937,63.922208,47.422197,10.302,201,0.055833
4,TIC 278352276,307.503625,48.607056,10.387,214,0.059444
5,TIC 283940788,8.851417,62.901594,11.774,448,0.124444
6,TIC 286470992,45.330708,60.572294,10.325,201,0.055833
7,TIC 307119043,14.827542,51.221642,9.940,181,0.050278
8,TIC 317863971,110.567500,3.031925,10.306,201,0.055833
9,TIC 322727163,309.716625,50.466819,10.997,274,0.076111
